In [1]:
# %% [markdown]
# ### 라이브러리 임포트 및 데이터 로드

import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

# CSV 파일 불러오기
file_path = "dataset.csv"
df = pd.read_csv(file_path)

# 불필요한 'Unnamed: 4' 컬럼 제거
if "Unnamed: 4" in df.columns:
    df = df.drop(columns=["Unnamed: 4"])

# title과 article을 합쳐서 새로운 컬럼 생성
df["text"] = df["title"] + " " + df["article"]

# 데이터 확인
print(df.head())
print("Unique categories:", df["category"].unique())


                            title category  \
0       서울대치과병원 갤러리 치유, 이돈아 초대개인전    IT/과학   
1          ‘불법도박’ 스팸, 2년 연속 최다 신고    IT/과학   
2  삼성과 애플도 손잡았다…삼성 스마트TV에 아이튠즈 적용    IT/과학   
3          NHN엔터 ‘한돌’ 프로기사 상대 4연승    IT/과학   
4     세란병원, 2018년 응급의료기관 평가 최우수등급    IT/과학   

                                             article sub_category  \
0  “행운, 행복, 부귀영화 상징…꽃 그림 보러 오세요”서울대치과병원 갤러리 치유, 이...     IT/과학 일반   
1  대출권유, 텔레마케팅 스팸신고도 많아지난해 총 1626만건 접수… 32% 늘어 한편...     IT/과학 일반   
2  삼성전자 영상디스플레이사업부 이원진 부사장은 “삼성전자는 OS나 제품 차이를 넘어서...     IT/과학 일반   
3  이제 남은 것은 국내 바둑 랭킹 1위(12월 기준) 신진서 9단과의 대국이다. 마지...     IT/과학 일반   
4  세란병원이 보건복지부에서 시행하는 ‘2018년 응급의료기관 평가’에서 최우수등급인 ...     IT/과학 일반   

                                                text  
0  서울대치과병원 갤러리 치유, 이돈아 초대개인전 “행운, 행복, 부귀영화 상징…꽃 그...  
1  ‘불법도박’ 스팸, 2년 연속 최다 신고 대출권유, 텔레마케팅 스팸신고도 많아지난해...  
2  삼성과 애플도 손잡았다…삼성 스마트TV에 아이튠즈 적용 삼성전자 영상디스플레이사업부...  
3  NHN엔터 ‘한돌’ 프로기사 상대 4연승 이제 남은 것은 국내 바둑 랭킹 1위(12...  
4  세란병원, 2018년 응급의료기관 평가 최우수등급 세란

In [10]:
# %% [markdown]
# ### 카테고리별 서브카테고리 매핑 (참고용)
# 아래 매핑은 모델이 예측할 sub_category가 입력받은 category 하위에 있어야 한다는 제약 조건을 명시하기 위한 정보야.
# (학습 데이터에는 이미 올바른 sub_category가 기록되어 있으므로, 모델 학습에는 별도 반영하지 않아도 됨)

category_to_subcategories = {
    "정치": ["정부와 정책", "국회와 법률", "선거와 정당"],
    "경제": ["금융과 투자", "산업과 기업", "시장과 물가", "부동산"],
    "사회": ["교육과 학교", "환경과 재해", "안전과 건강", "노동과 시민사회", "사회 일반"],
    "생활/문화": ["문화와 예술", "여가와 생활"],
    "IT/과학": ["인공지능과 로봇", "디지털 기술과 인터넷", "우주와 자연 과학", "IT/과학 일반"],
    "세계": ["외교와 글로벌 경제", "국제 사회와 전쟁", "세계 문화와 생활"],
    "스포츠": ["축구", "야구", "농구", "배구", "올림픽과 국제대회", "스포츠 일반"]
}

# 카테고리명 영어 매핑
category_name_map = {
    "정치": "politics",
    "경제": "economy",
    "사회": "society",
    "생활/문화": "life",
    "IT/과학": "it_science",
    "세계": "world",
    "스포츠": "sports"
}


In [11]:
from sklearn.metrics import classification_report, accuracy_score
import os
import joblib

# 전체 y_true, y_pred 누적 저장용 리스트
all_y_true = []
all_y_pred = []

# 저장 디렉토리 만들기
os.makedirs("saved_models", exist_ok=True)

category_models = {}

for category in df["category"].unique():
    subset = df[df["category"] == category]
    X = subset["text"].values
    y = subset["sub_category"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000)),
        ('clf', LogisticRegression(max_iter=1000))
    ])

    pipeline.fit(X_train, y_train)
    category_models[category] = pipeline

    # ✅ 모델 저장
    safe_category_name = category_name_map.get(category, category)
    model_path = f"saved_models/{safe_category_name}_pipeline.pkl"
    joblib.dump(pipeline, model_path)
    
    y_pred = pipeline.predict(X_test)

    # 개별 출력
    print(f"Category: {category}")
    print(classification_report(y_test, y_pred))
    print("=" * 50)

    # 전체 누적
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)

# 전체 성능 출력
print("==== Overall Performance ====")
print(classification_report(all_y_true, all_y_pred))
print("Accuracy:", accuracy_score(all_y_true, all_y_pred))


Category: IT/과학
              precision    recall  f1-score   support

    IT/과학 일반       0.74      0.67      0.70        42
 디지털 기술과 인터넷       0.67      0.64      0.65        47
   우주와 자연 과학       1.00      1.00      1.00        54
    인공지능과 로봇       0.67      0.76      0.71        46

    accuracy                           0.78       189
   macro avg       0.77      0.77      0.77       189
weighted avg       0.78      0.78      0.78       189

Category: 경제
              precision    recall  f1-score   support

      금융과 투자       0.73      0.78      0.76        46
      소비와 물가       0.79      0.74      0.76        50

    accuracy                           0.76        96
   macro avg       0.76      0.76      0.76        96
weighted avg       0.76      0.76      0.76        96

Category: 사회
              precision    recall  f1-score   support

      교육과 학교       0.63      0.79      0.70        42
       사회 일반       0.69      0.66      0.67        53
      안전과 건강       0.65      0.69